In [1]:
import pandas as pd
import warnings
warnings.filterwarnings("ignore")
dataset = pd.read_csv('50_Startups.csv')
dataset

,R&D Spend,Administration,Marketing Spend,State,Profit
0,165349.20,136897.80,471784.10,New York,192261.83
1,162597.70,151377.59,443898.53,California,191792.06
2,153441.51,101145.55,407934.54,Florida,191050.39
3,144372.41,118671.85,383199.62,New York,182901.99
4,142107.34,91391.77,366168.42,Florida,166187.94
5,131876.90,99814.71,362861.36,New York,156991.12
6,134615.46,147198.87,127716.82,California,156122.51
7,130298.13,145530.06,323876.68,Florida,155752.60
8,120542.52,148718.95,311613.29,New York,152211.77
9,123334.88,108679.17,304981.62,California,149759.96


In [3]:
dataset = pd.get_dummies(dataset,dtype=int,drop_first=True)
dataset.columns
independent = dataset[['R&D Spend', 'Administration', 'Marketing Spend','State_Florida', 'State_New York' ]]
independent
dependent = dataset[['Profit']]
dependent

,Profit
0,192261.83
1,191792.06
2,191050.39
3,182901.99
4,166187.94
5,156991.12
6,156122.51
7,155752.60
8,152211.77
9,149759.96


In [4]:
from sklearn.model_selection import train_test_split
x_train,x_test,y_train,y_test = train_test_split(independent,dependent,test_size =0.3,random_state =0)

In [5]:
# standarised method
from sklearn.preprocessing import StandardScaler
sc = StandardScaler()
x_train = sc.fit_transform(x_train)
x_test = sc.transform(x_test)
x_train

scy = StandardScaler()
y_train = scy.fit_transform(y_train)
y_test = scy.transform(y_test)
y_test

array([[-0.16141583],
       [ 0.79125539],
       [ 0.83455765],
       [-0.75388069],
       [ 1.87909509],
       [-0.12128983],
       [-0.67413157],
       [-0.2962321 ],
       [ 0.00295097],
       [ 1.30107013],
       [-0.31261421],
       [-0.31957517],
       [-0.10442902],
       [-0.31415143],
       [ 0.32645147]])

In [6]:
# cv method
from sklearn.model_selection import GridSearchCV
from sklearn.svm import SVR

param_grid ={'kernel':['rdf','poly','sigmoid','linear'],'C':[10,100,1000,2000,3000],'gamma':['auto','scale']}

grid = GridSearchCV(SVR(),param_grid,refit=True,verbose =3,n_jobs=-1)
grid.fit(x_train,y_train)
grid



Fitting 5 folds for each of 40 candidates, totalling 200 fits


,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",SVR()
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'C': [10, 100, ...], 'gamma': ['auto', 'scale'], 'kernel': ['rdf', 'poly', ...]}"
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"verbose verbose: int, default=0Controls the verbosity of information printed during fitting, with highervalues yielding more detailed logging.- 0 : no messages are printed;- >=1 : summary of the total number of fits;- >=2 : computation time for each fold and parameter candidate;- >=3 : fold indices and scores;- >=10 : parameter candidate indices and START messages before each fit.",3
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion <scoring_api_overview>` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",None
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example<sphx_glr_auto_examples_model_selection_plot_grid_search_refit_callable.py>`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- an iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :cl

In [7]:
print(grid.best_params_)

{'C': 100, 'gamma': 'auto', 'kernel': 'linear'}


In [8]:
re = grid.cv_results_
re
table=pd.DataFrame.from_dict(re)
table

,mean_fit_time,std_fit_time,mean_score_time,std_score_time,param_C,param_gamma,param_kernel,params,split0_test_score,split1_test_score,split2_test_score,split3_test_score,split4_test_score,mean_test_score,std_test_score,rank_test_score
0,0.002185,0.000665,0.000000,0.000000,10,auto,rdf,"{'C': 10, 'gamma': 'auto', 'kernel': 'rdf'}",NaN,NaN,NaN,NaN,NaN,NaN,NaN,31
1,0.006855,0.002952,0.004100,0.001249,10,auto,poly,"{'C': 10, 'gamma': 'auto', 'kernel': 'poly'}",-9.640903e-01,8.945918e-01,0.730909,3.806707e-01,0.572856,3.229875e-01,6.656519e-01,12
2,0.005045,0.001112,0.003963,0.000907,10,auto,sigmoid,"{'C': 10, 'gamma': 'auto', 'kernel': 'sigmoid'}",-2.137841e+01,-1.053629e+01,-7.083174,-6.473032e+01,-1.885680,-2.112277e+01,2.271974e+01,22
3,0.016180,0.003476,0.004000,0.000621,10,auto,linear,"{'C': 10, 'gamma': 'auto', 'kernel': 'linear'}",8.585589e-01,9.473769e-01,0.919079,8.318672e-01,0.952253,9.018271e-01,4.833572e-02,3
4,0.002336,0.001067,0.000000,0.000000,10,scale,rdf,"{'C': 10, 'gamma': 'scale', 'kernel': 'rdf'}",NaN,NaN,NaN,NaN,NaN,NaN,NaN,31
5,0.006121,0.000323,0.003749,0.000509,10,scale,poly,"{'C': 10, 'gamma': 'scale', 'kernel': 'poly'}",-7.657603e-01,8.845850e-01,0.727028,4.066442e-01,0.601611,3.708215e-01,5.894406e-01,11
6,0.005454,0.001547,0.003272,0.000603,10,scale,sigmoid,"{'C': 10, 'gamma': 'scale', 'kernel': 'sigmoid'}",-1.473106e+01,-4.940095e+00,-6.147813,-6.678368e+01,-2.415308,-1.900359e+01,2.424630e+01,21
7,0.012487,0.002489,0.003229,0.000470,10,scale,linear,"{'C': 10, 'gamma': 'scale', 'kernel': 'linear'}",8.585589e-01,9.473769e-01,0.919079,8.318672e-01,0.952253,9.018271e-01,4.833572e-02,3
8,0.001533,0.000231,0.000000,0.000000,100,auto,rdf,"{'C': 100, 'gamma': 'auto', 'kernel': 'rdf'}",NaN,NaN,NaN,NaN,NaN,NaN,NaN,31
9,0.007807,0.002413,0.003049,0.000689,100,auto,poly,"{'C': 100, 'gamma': 'auto', 'kernel': 'poly'}",-1.163642e+00,7.934828e-01,0.747542,4.450937e-01,0.719070,3.083092e-01,7.459751e-01,13


In [9]:
from sklearn.metrics import r2_score

y_pred = grid.predict(x_test)
r_score=r2_score(y_test, y_pred)
print("R2 Score:", r2_score(y_test, y_pred))

R2 Score: 0.937500502733674


In [10]:
user_age = float(input("Enter Age: "))
user_bmi = float(input("Enter BMI: "))
user_children = float(input("Enter Number of Children: "))
user_sex = float(
    input("Enter Sex (1 for Male, 0 for Female): ")
)  # Male = 1, Female = 0
user_smoker = float(
    input("Enter Smoker (1 for Yes, 0 for No): ")
) 

Enter Age:  60
Enter BMI:  56.89
Enter Number of Children:  6
Enter Sex (1 for Male, 0 for Female):  1
Enter Smoker (1 for Yes, 0 for No):  0


In [11]:
import pickle
# 5. Save all 3 required files using pickle
pickle.dump(grid, open('cv_model.sav', 'wb'))
pickle.dump(sc, open('scaler_x.sav', 'wb'))
pickle.dump(scy, open('scaler_y.sav', 'wb'))
# 1. Load saved model and scalers
loaded_model = pickle.load(open('cv_model.sav', 'rb'))
loaded_sc = pickle.load(open('scaler_x.sav', 'rb'))
loaded_scy = pickle.load(open('scaler_y.sav', 'rb'))

In [12]:
# 3. Transform inputs using loaded feature scaler
user_data = [[
    user_age,
    user_bmi,
    user_children,
    user_sex,
    user_smoker,
]]
scaled_input = loaded_sc.transform(user_data)

# 4. Predict scaled output
scaled_pred = loaded_model.predict(scaled_input)

# 5. Inverse transform prediction back to original scale (Profit in $)
# .reshape(-1, 1) converts 1D output back to 2D for inverse_transform
final_profit = loaded_scy.inverse_transform(scaled_pred.reshape(-1, 1))
final_profit
# print(f'\nPredicted Profit: ${final_profit[0][0]:,.2f}')

array([[45336.29085524]])